# Cobre 2010–2025: Cochilco como fuente principal
Decisión adoptada el 16-10-2026: usar exclusivamente el precio diario spot BML/LME de Cochilco para todo el período del proyecto. El Excel BCCh se conserva como referencia; no se mezclan las dos fuentes.

El notebook conserva primero la investigación de la historia faltante y las discrepancias. La sección 6 construye la **tabla principal para futuros análisis**: `data/processed/cobre_cochilco_diario_2010_2025.csv`. Todo el procesamiento usa archivos locales.

Las descargas originales y sus parámetros y huellas están en `data/raw/cobre_historico/`. No se modifica el notebook de limpieza del compañero.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/processing/cobre_historico.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.processing.cobre_historico import build, read_bcentral, main
import pandas as pd
import json

procedencia = json.loads((ROOT / 'data/raw/cobre_historico/procedencia.json').read_text(encoding='utf-8'))
print(procedencia['source'])
print(procedencia['download_url'])
print(procedencia['parameters'])

Comisión Chilena del Cobre (Cochilco)
https://boletin.cochilco.cl/estadisticas/grafico/crear_excel.asp
{'tipo_metal': 'Cobre', 'tipo_consulta': '1', 'bolsa_final': 'BML', 'dia': '1', 'mes': '1', 'anio': '2010', 'dia1': '31', 'mes1': '12', 'anio1': '2013', 'bandera': 'S'}


## 1. Por qué faltaba el comienzo del período
La serie diaria BCCh `F019.PPB.PRE.100.D` informa su primera observación el **05-10-2012** en el [buscador oficial](https://si3.bcentral.cl/Search/Buscar/Index/?st=F019.PPB.PRE.100.D&Idioma=ES). El Excel comienza ese mismo día. El selector del cuadro BDE admite 2010 porque también muestra otras materias primas; eso no significa que cobre tenga valores en ese año. No hay evidencia de una descarga incompleta del usuario.

In [2]:
bcch = read_bcentral(ROOT / 'data/raw/cu_usd_lb.xlsx')
print('BCCh:', bcch.fecha.min().date(), 'a', bcch.fecha.max().date())
print(bcch.head().to_string(index=False))

BCCh: 2012-10-05 a 2026-09-23
     fecha  cu_usd_lb
2012-10-05       3.76
2012-10-08       3.76
2012-10-09       3.71
2012-10-10       3.69
2012-10-11       3.70


## 2. Recuperar observaciones, sin inventar fechas
La consulta de Cochilco selecciona **Cobre, diaria, BML/LME**, desde 01-01-2010 hasta 31-12-2013. Los años adicionales permiten comparar fuentes. El archivo conserva el HTML original que el servidor entrega con extensión `.xls`.

La fuente expresa **centavos de dólar por libra**: se divide por 100 para obtener USD/libra. No se convierten números a fechas, ni se interpolan festivos o fines de semana. El primer precio recuperado es 338,56 centavos/libra = 3,3856 USD/libra. La tabla 2.2 de Cochilco para enero de 2010 lo identifica como spot BML (338,561 con mayor precisión).

In [3]:
cochilco, faltante, comparacion, resumen = build(ROOT)
print('Cotizaciones recuperadas del tramo faltante:', len(faltante))
print(faltante.head().to_string(index=False))
print(faltante.tail().to_string(index=False))
print('Observaciones por año del suplemento:')
print(faltante.groupby(faltante.fecha.dt.year).size().to_string())

Cotizaciones recuperadas del tramo faltante: 696
     fecha  cu_usd_lb   fuente               serie_origen    unidad
2010-01-04     3.3856 Cochilco Cobre BML, consulta diaria USD/libra
2010-01-05     3.3729 Cochilco Cobre BML, consulta diaria USD/libra
2010-01-06     3.4416 Cochilco Cobre BML, consulta diaria USD/libra
2010-01-07     3.4446 Cochilco Cobre BML, consulta diaria USD/libra
2010-01-08     3.3774 Cochilco Cobre BML, consulta diaria USD/libra
     fecha  cu_usd_lb   fuente               serie_origen    unidad
2012-09-28     3.7501 Cochilco Cobre BML, consulta diaria USD/libra
2012-10-01     3.7156 Cochilco Cobre BML, consulta diaria USD/libra
2012-10-02     3.7830 Cochilco Cobre BML, consulta diaria USD/libra
2012-10-03     3.7539 Cochilco Cobre BML, consulta diaria USD/libra
2012-10-04     3.7762 Cochilco Cobre BML, consulta diaria USD/libra
Observaciones por año del suplemento:
fecha
2010    253
2011    251
2012    192


## 3. Comparar antes de integrar
Se cruza por fecha exacta, sin desplazar días ni ajustar precios para acercar las fuentes. En el intervalo común el BCCh entrega valores con dos decimales: ±0,005 USD/libra es el margen de redondeo usado para el control. Una correlación alta no basta para declarar que dos series son intercambiables.

In [4]:
print(json.dumps(resumen, ensure_ascii=False, indent=2))
print('Primeras fechas comunes:')
print(comparacion.head(12).to_string(index=False))
print('Mayores discrepancias absolutas:')
indices = comparacion.diferencia_usd_lb.abs().nlargest(5).index
print(comparacion.loc[indices].to_string(index=False))

{
  "cochilco_observaciones": 1009,
  "faltante_recuperado_observaciones": 696,
  "faltante_desde": "2010-01-04",
  "faltante_hasta": "2012-10-04",
  "primera_observacion_bcentral": "2012-10-05",
  "comparacion_observaciones": 299,
  "comparacion_desde": "2012-10-05",
  "comparacion_hasta": "2013-12-30",
  "diferencia_absoluta_media_usd_lb": 0.033824692307692306,
  "diferencia_absoluta_maxima_usd_lb": 0.21019999999999994,
  "coinciden_dentro_redondeo": 39,
  "correlacion": 0.973769179340497,
  "empalme_automatico": false,
  "motivo": "Se recupera la historia, pero las fuentes no coinciden solo por redondeo. Conservar por separado hasta resolver la definición temporal/metodológica."
}
Primeras fechas comunes:
     fecha  cu_usd_lb_bcentral  cu_usd_lb_cochilco  diferencia_usd_lb  compatible_solo_redondeo
2012-10-05                3.76              3.7644             0.0044                      True
2012-10-08                3.76              3.6891            -0.0709                     

## 4. Guardar resultados regenerables
Se generan tres tablas: descarga convertida a USD/libra, suplemento anterior al inicio de BCCh y comparación de fechas comunes. No se sobrescribe el Excel original ni el CSV del compañero. El informe de validación queda en `docs/cobre_validacion.json`. El código verifica la huella de la descarga, el identificador BCCh, duplicados y valores.

In [5]:
main()
for name in ('cobre_cochilco_diario_2010_2013.csv', 'cu_usd_lb_faltante_2010_2012.csv', 'cobre_comparacion_2012_2013.csv'):
    output = pd.read_csv(ROOT / 'data/processed' / name)
    print(name, '→', len(output), 'filas')

{
  "cochilco_observaciones": 1009,
  "faltante_recuperado_observaciones": 696,
  "faltante_desde": "2010-01-04",
  "faltante_hasta": "2012-10-04",
  "primera_observacion_bcentral": "2012-10-05",
  "comparacion_observaciones": 299,
  "comparacion_desde": "2012-10-05",
  "comparacion_hasta": "2013-12-30",
  "diferencia_absoluta_media_usd_lb": 0.033824692307692306,
  "diferencia_absoluta_maxima_usd_lb": 0.21019999999999994,
  "coinciden_dentro_redondeo": 39,
  "correlacion": 0.973769179340497,
  "empalme_automatico": false,
  "motivo": "Se recupera la historia, pero las fuentes no coinciden solo por redondeo. Conservar por separado hasta resolver la definición temporal/metodológica."
}
cobre_cochilco_diario_2010_2013.csv → 1009 filas
cu_usd_lb_faltante_2010_2012.csv → 696 filas
cobre_comparacion_2012_2013.csv → 299 filas


## 5. Decisión de fuente y alcance temporal
El equipo adoptó Cochilco para todo 2010–2025. La tabla BCCh queda como referencia histórica, sin empalme ni ajuste forzado de precios. La causa exacta de sus discrepancias permanece sin demostrar.

Se mantiene el corte de predicción **al cierre del día hábil anterior a la decisión de TPM**. La fecha de cotización no certifica su hora de disponibilidad histórica; ese control se realizará antes de construir predictores.

Detalles y enlaces oficiales: [`docs/cobre_historico.md`](../docs/cobre_historico.md).

## 6. Construir Cochilco 2010–2025: serie elegida
Se conservan tres exportaciones originales: 2010–2013, 2013–2019 y 2019–2025. Se verifica su SHA-256 y selección de Cobre/Diaria/BML. Los años repetidos permiten comprobar coincidencia exacta antes de eliminar duplicados. Un precio contradictorio detiene el proceso.

No se consulta Internet, no se interpola y no se reemplazan cotizaciones con BCCh. La unidad de la descarga es centavos/libra y se divide por 100 para obtener USD/libra.


In [6]:
from src.processing.cobre_cochilco import build as construir_serie_completa, main as guardar_serie_completa

cobre, calendario_sin_dato, validacion_completa = construir_serie_completa(ROOT)
print('Observaciones:', len(cobre))
print('Cobertura:', cobre.fecha.min().date(), 'a', cobre.fecha.max().date())
print('Meses cubiertos:', validacion_completa['meses_cubiertos'])
print('Duplicados idénticos eliminados:', validacion_completa['duplicados_identicos_eliminados'])
print(cobre.head().to_string(index=False))
print(cobre.tail().to_string(index=False))
print(cobre.groupby(cobre.fecha.dt.year).size().to_string())


Observaciones: 4042
Cobertura: 2010-01-04 a 2025-12-31
Meses cubiertos: 192
Duplicados idénticos eliminados: 506
     fecha  cu_usd_lb   fuente      mercado    unidad                   archivo_origen
2010-01-04     3.3856 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2010_2013.xls
2010-01-05     3.3729 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2010_2013.xls
2010-01-06     3.4416 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2010_2013.xls
2010-01-07     3.4446 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2010_2013.xls
2010-01-08     3.3774 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2010_2013.xls
     fecha  cu_usd_lb   fuente      mercado    unidad                   archivo_origen
2025-12-23     5.4749 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-24     5.5579 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-29     5.5819 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-30     5.

fecha
2010    253
2011    251
2012    252
2013    253
2014    253
2015    253
2016    253
2017    252
2018    253
2019    253
2020    254
2021    253
2022    251
2023    251
2024    254
2025    253


## 7. Calendario y controles
Se verifican fechas únicas, precios positivos, cobertura de todos los meses y recuentos anuales. Los días lunes a viernes sin observación se guardan como inventario para revisión; incluyen feriados/cierres de mercado y no se rellenan. Este control general no certifica individualmente todos los festivos de Londres.


In [7]:
print('Fechas duplicadas:', int(cobre.fecha.duplicated().sum()))
print('Precios nulos:', int(cobre.cu_usd_lb.isna().sum()))
print('Fuentes:', cobre.fuente.unique().tolist())
print('Lunes a viernes sin observación:', len(calendario_sin_dato))
print(calendario_sin_dato.head(12).to_string(index=False))
print('Rango de precios USD/libra:', cobre.cu_usd_lb.min(), cobre.cu_usd_lb.max())


Fechas duplicadas: 0
Precios nulos: 0
Fuentes: ['Cochilco']
Lunes a viernes sin observación: 132
     fecha                                                        estado
2010-01-01 Sin observación en la fuente; revisar calendario, no imputado
2010-04-02 Sin observación en la fuente; revisar calendario, no imputado
2010-04-05 Sin observación en la fuente; revisar calendario, no imputado
2010-05-03 Sin observación en la fuente; revisar calendario, no imputado
2010-05-31 Sin observación en la fuente; revisar calendario, no imputado
2010-08-30 Sin observación en la fuente; revisar calendario, no imputado
2010-12-27 Sin observación en la fuente; revisar calendario, no imputado
2010-12-28 Sin observación en la fuente; revisar calendario, no imputado
2011-01-03 Sin observación en la fuente; revisar calendario, no imputado
2011-04-22 Sin observación en la fuente; revisar calendario, no imputado
2011-04-25 Sin observación en la fuente; revisar calendario, no imputado
2011-04-29 Sin observación 

## 8. Guardar e importar para futuros análisis
La entrada recomendada es `cobre_cochilco_diario_2010_2025.csv`. Las tablas del suplemento y comparación anteriores quedan como auditoría. `archivo_origen` permite rastrear cada fila hasta una descarga conservada.

Los archivos se escriben en `data/processed/` y el informe en `docs/cobre_cochilco_validacion.json`. Todavía no se construyen etiquetas, rezagos ni una base de entrenamiento.


In [8]:
guardar_serie_completa()
cobre_para_analisis = pd.read_csv(
    ROOT / 'data/processed/cobre_cochilco_diario_2010_2025.csv',
    parse_dates=['fecha'],
)
assert len(cobre_para_analisis) == len(cobre)
assert cobre_para_analisis.fuente.eq('Cochilco').all()
print(cobre_para_analisis.dtypes.to_string())
print(cobre_para_analisis.tail().to_string(index=False))


{
  "fuente_elegida": "Cochilco",
  "mercado": "BML/LME spot",
  "unidad": "USD/libra",
  "periodo_proyecto": "2010–2025",
  "observaciones": 4042,
  "primera_fecha": "2010-01-04",
  "ultima_fecha": "2025-12-31",
  "meses_cubiertos": 192,
  "duplicados_identicos_eliminados": 506,
  "observaciones_por_anio": {
    "2010": 253,
    "2011": 251,
    "2012": 252,
    "2013": 253,
    "2014": 253,
    "2015": 253,
    "2016": 253,
    "2017": 252,
    "2018": 253,
    "2019": 253,
    "2020": 254,
    "2021": 253,
    "2022": 251,
    "2023": 251,
    "2024": 254,
    "2025": 253
  },
  "dias_lunes_viernes_sin_observacion": 132,
  "observaciones_fin_de_semana": 0,
  "precio_minimo_usd_lb": 1.9552,
  "precio_maximo_usd_lb": 5.6753,
  "mezcla_con_bcentral": false,
  "interpolacion": false,
  "relleno_de_fechas": false,
  "fecha_significado": "Fecha de cotización informada por la fuente; no certifica hora de publicación.",
  "archivos_fuente": [
    {
      "archivo": "cochilco_cobre_bml_2010_

fecha             datetime64[us]
cu_usd_lb                float64
fuente                       str
mercado                      str
unidad                       str
archivo_origen               str
     fecha  cu_usd_lb   fuente      mercado    unidad                   archivo_origen
2025-12-23     5.4749 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-24     5.5579 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-29     5.5819 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-30     5.6753 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
2025-12-31     5.6717 Cochilco BML/LME spot USD/libra cochilco_cobre_bml_2019_2025.xls
